# è fortemente consigliato runnare il notebook su colab

# Preliminari

In [ ]:
%%capture
!pip uninstall -y unsloth peft

!pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
import torch
import json
import copy
import os

from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import train_on_responses_only
from pathlib import Path
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from transformers import TrainingArguments
from transformers import TextStreamer
from huggingface_hub import login
from google.colab import userdata

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

In [ ]:
# For GPU check
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')

CUDA available: True
GPU: Tesla T4


# Parametri

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
model_name = "unsloth/Phi-4"
TRAIN_FILE_NAME = "data_luca_train-v1.jsonl"
TEST_FILE_NAME = "data_luca_test-v1.jsonl"
VAL_FILE_NAME = "data_luca_val-v1.jsonl"
#CHAT_TEMPLATE = "llama-3.1"
FINETUNED_MODEL_NAME = "phi-4-v2"
RANDOM_STATE = 2025

In [ ]:
FINETUNED_MODEL_NAME = "phi-4-v2"

# Load model and tokenizer

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

==((====))==  Unsloth 2025.10.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.39G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Add Lora adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA rank. Choose any number > 0 ! Suggested 8, 16, 32, 64, 128. Higher, more capacity, more memory.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # LoRA scaling factor (usually 2x rank)
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = RANDOM_STATE,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.10.1 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


# Data preparation

In [ ]:

# Carichiamo il nostro file JSON
if "COLAB_" not in "".join(os.environ.keys()):
    train_path = Path('../data/ft-dataset/' + TRAIN_FILE_NAME)
    test_path = Path('../data/ft-dataset/' + TEST_FILE_NAME)
    if VAL_FILE_NAME not in ["", None]:
        val_path = Path('../data/ft-dataset/' + VAL_FILE_NAME)
else:
    train_path = TRAIN_FILE_NAME
    test_path = TEST_FILE_NAME
    if VAL_FILE_NAME not in ["", None]:
        val_path = VAL_FILE_NAME

with open(train_path, "r", encoding="utf-8") as f:
    raw_train = [json.loads(line) for line in f]

with open(test_path, "r", encoding="utf-8") as f:
    raw_test = [json.loads(line) for line in f]

if VAL_FILE_NAME not in ["", None]:
    with open(val_path, "r", encoding="utf-8") as f:
        raw_val = [json.loads(line) for line in f]

I dati in `raw_train` sono salvati con la seguente struttura:
```
[
    {
        "messages": [
            {"role": "system", "content": <contenuto di sistema>},
            {"role": "user", "content": <contenuto di user>},
            {"role": "assistant", "content": <contenuto di assistant>}
        ]
    },
    ...
]
```

## Creazione dataset corretto

Trasformiamo `<contenuto di assistant>` in una stringa.

In [ ]:
training_data = copy.deepcopy(raw_train)
test_data = copy.deepcopy(raw_test)
val_data = copy.deepcopy(raw_val)

for split in [training_data, test_data, val_data]:
    if split is None:
        continue
    for conversation in split:
        for message in conversation['messages']:
            if message['role'] == 'assistant':
                message['content'] = json.dumps(message['content'])

In [ ]:
print(type(raw_train[0]['messages'][2]['content']), raw_train[0]['messages'][2]['content'])
print(type(training_data[0]['messages'][2]['content']), training_data[0]['messages'][2]['content'])

<class 'dict'> {'morfologia': 'solido_anulare', 'spessore_parietale': None, 'estensione_cranio_caudale': None, 'distanza_oai': None, 'posizione': 'giunzione', 'carcinosi_peritoneale': None, 'lesioni_ossee': None, 'riflessione_peritoneale_anteriore': 'cavallo', 'infiltrazione_tessuto_adiposo': 'si_5mm_plus', 'infiltrazione_sfinteri': 'no', 'infiltrazione_organi_extra': 'no', 'coinvolgimento_riflessione_peritoneale': 'no', 'coinvolgimento_fascia_mesorettale': 'no', 'linfonodi_sospetti': 10.0, 'linfonodi_mesorettali': True, 'linfonodi_rettali_superiori': True, 'linfonodi_mesenterici_inferiori': False, 'linfonodi_iliaci_interni': True, 'linfonodi_otturatori': True, 'linfonodi_sacrali': False, 'linfonodi_inguinali_sotto_dentata': False, 'linfonodi_inguinali': False, 'linfonodi_iliaci_esterni': True, 'linfonodi_iliaci_comuni': True, 'linfonodi_paraortici': False, 'linfonodi_altri': False, 'depositi_tumorali': 'no', 'numero_depositi': 0.0, 'emvi_esteso': 'no'}
<class 'str'> {"morfologia": "so

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [ ]:
def formatting_messages(messages: list[dict]) -> str:
    return tokenizer.apply_chat_template(messages,
                                         tokenize=False,
                                         add_generation_prompt=False)

In [ ]:
def create_hugging_face_dataset(data: list[dict]) -> Dataset:
    system_content = []
    user_content = []
    assistant_content = []
    formatted_text = []
    for conversation in data:
        formatted_text.append(formatting_messages(conversation['messages']))
        for message in conversation['messages']:
            if message['role'] == 'system':
                system_content.append(message['content'])
            elif message['role'] == 'user':
                user_content.append(message['content'])
            elif message['role'] == 'assistant':
                assistant_content.append(message['content'])
    return Dataset.from_dict(
            {
                'system_content': system_content,
                'user_content': user_content,
                'assistant_content': assistant_content,
                'text': formatted_text
            })

In [ ]:
dataset_train = create_hugging_face_dataset(training_data)
dataset_test = create_hugging_face_dataset(test_data)
if VAL_FILE_NAME not in ["", None]:
    dataset_val = create_hugging_face_dataset(val_data)

In [ ]:
for split in [dataset_train, dataset_test, dataset_val]:
    if split is None:
        continue
    display(split)

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 120
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 18
})

Vediamo come il chat template ha trasformato le conversazioni.

In [ ]:
print(dataset_train[0]['text'])

<|im_start|>system<|im_sep|>Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è analizzare attentamente il referto di Risonanza Magnetica fornito e mapparlo nel seguente schema JSON.

Regole di output rigorose:
1. Ogni campo deve contenere ESCLUSIVAMENTE uno dei valori previsti dalle liste/tipologie indicate.
2. Se un dato non è riportato o è ambiguo, inserisci la parola chiave "null" (minuscola).
3. Per i linfonodi, restituisci chiavi booleane ("true"/"false") per ogni sede possibile.
4. NON aggiungere testo, spiegazioni, preamboli o commenti.
5. Utilizza tutte le chiavi fornite nello schema JSON.
6. Restituisci ESCLUSIVAMENTE il JSON finale, assicurandoti che il formato sia VALIDO.

Schema JSON per l'output:

{
  "morfologia": "solido_polipoide | solido_anulare | mucinoso",
  "posizione": "basso | medio | alto | giunzione",
  "spessore_parietale": "int o null",
  "estensione_cranio_caudale": "int o n

<a name="Train"></a>
# Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    dataset_text_field = "text",
    eval_dataset=dataset_val,
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = RANDOM_STATE,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc,
        eval_strategy='steps',
        push_to_hub=True,
        hub_model_id=FINETUNED_MODEL_NAME
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/120 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/18 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [ ]:
print(len(trainer.train_dataset[0]['input_ids']), len(trainer.train_dataset[2]['input_ids']))

1998 1415


In [ ]:
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user<|im_sep|>",
    response_part="<|im_start|>assistant<|im_sep|>",
)

Map (num_proc=2):   0%|          | 0/120 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/18 [00:00<?, ? examples/s]

In [ ]:
trainer.train_dataset

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 120
})

We verify masking is actually done:

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]  # id di sapce
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

'                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

We can see the System and Instruction prompts are successfully masked!

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
10.727 GB of memory reserved.


In [ ]:
# @title Training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 65,536,000 of 14,725,043,200 (0.45% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.403900
2,0.402200
3,0.402200
4,0.372200
5,0.272000
6,0.178700
7,0.125600
8,0.106500
9,0.088900
10,0.073700


In [ ]:
# @title Push model to the HUB
trainer.push_to_hub("End of training")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...outputs/training_args.bin: 100%|##########| 6.22kB / 6.22kB            

  ...adapter_model.safetensors:  10%|9         | 25.0MB /  262MB            

CommitInfo(commit_url='https://huggingface.co/iltramont/phi-4-v2/commit/0b9dc61f7290d1d83b3bcd12d10a0ae45bb7c5c2', commit_message='End of training', commit_description='', oid='0b9dc61f7290d1d83b3bcd12d10a0ae45bb7c5c2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iltramont/phi-4-v2', endpoint='https://huggingface.co', repo_type='model', repo_id='iltramont/phi-4-v2'), pr_revision=None, pr_num=None)

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

1901.1279 seconds used for training.
31.69 minutes used for training.
Peak reserved memory = 14.086 GB.
Peak reserved memory for training = 3.359 GB.
Peak reserved memory % of max memory = 95.557 %.
Peak reserved memory for training % of max memory = 22.787 %.


In [ ]:
trainer_stats

TrainOutput(global_step=30, training_loss=0.1283542474110921, metrics={'train_runtime': 1901.1279, 'train_samples_per_second': 0.126, 'train_steps_per_second': 0.016, 'total_flos': 3.227099966189568e+16, 'train_loss': 0.1283542474110921, 'epoch': 2.0})

<a name="Inference"></a>
# Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
test_messages = [
    {'role': 'system', 'content': dataset_test[0]['system_content']},
    {'role': 'user', 'content': dataset_test[0]['user_content']}
]

In [ ]:
tokenized_chat_template = tokenizer.apply_chat_template(
                        test_messages,
                        tokenize = True,
                        add_generation_prompt = True, # Must add for generation
                        return_tensors = "pt"
                        ).to("cuda")

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(100352, 5120, padding_idx=100351)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [ ]:
outputs = model.generate(input_ids=tokenized_chat_template,
                         use_cache=True,
                         temperature=1.5,
                         min_p=0.1)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
generated_tokens = outputs[0][len(tokenized_chat_template[0]):-1]
print(tokenizer.decode(generated_tokens))

{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 50.0, "distanza_oai": 60.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "sedi_locoregionali": {"mesorettali": true, "rettali_superiori": false, "mesenterici_inferiori": false, "iliaci_interni": false, "otturatori": false, "sacrali": false, "inguinali_sotto_dentata": false}, "sedi_non_locoregionali": {"inguinali": false, "iliaci_esterni": false, "iliaci_comuni": false, "paraortici": false, "altri": false}, "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}


In [ ]:
dataset_test

Dataset({
    features: ['system_content', 'user_content', 'assistant_content', 'text'],
    num_rows: 35
})

In [ ]:
prediction = json.loads(tokenizer.decode(generated_tokens))
groundtruth = groundtruth = json.loads(dataset_test[0]['assistant_content'])

print(f"{'LABELS':.^40} {'TRUTH':.^18} {'PREDICTION':.^18}")
for key in groundtruth.keys():
    if key not in ('sedi_locoregionali', 'sedi_non_locoregionali'):
        print(f"{key:.<40} {str(groundtruth[key]):.^18} {str(prediction[key]):.^18}")
    else:
        print(f"{key}")
        for subkey in groundtruth[key].keys():
            print(5*' ' + f"{subkey:.<35} {str(groundtruth[key][subkey]):.^18} {str(prediction[key][subkey]):.^18}")

.................LABELS................. ......TRUTH....... ....PREDICTION....
morfologia.............................. ..solido_anulare.. .solido_polipoide.
spessore_parietale...................... .......None....... .......None.......
estensione_cranio_caudale............... .......50.0....... .......50.0.......
distanza_oai............................ .......60.0....... .......60.0.......
posizione............................... ......medio....... ......medio.......
carcinosi_peritoneale................... ........no........ ........no........
lesioni_ossee........................... ........no........ ........no........
riflessione_peritoneale_anteriore....... .......None....... .....cavallo......
infiltrazione_tessuto_adiposo........... ...si_5mm_plus.... ...si_5mm_plus....
infiltrazione_sfinteri.................. ........no........ ........no........
infiltrazione_organi_extra.............. ........no........ ........no........
coinvolgimento_riflessione_peritoneale.. ........si.

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=tokenized_chat_template,
                   streamer=text_streamer,
                   use_cache=True,
                   temperature=1.5,
                   min_p=0.1)

{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 50.0, "distanza_oai": 60.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "sedi_locoregionali": {"mesorettali": true, "rettali_superiori": false, "mesenterici_inferiori": false, "iliaci_interni": false, "otturatori": false, "sacrali": false, "inguinali_sotto_dentata": false}, "sedi_non_locoregionali": {"inguinali": false, "iliaci_esterni": false, "iliaci_comuni": false, "paraortici": false, "altri": false}, "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}<|im_end|>


<a name="Save"></a>
# Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained(FINETUNED_MODEL_NAME)  # Local saving
tokenizer.save_pretrained(FINETUNED_MODEL_NAME)
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('phi-4-v1/tokenizer_config.json',
 'phi-4-v1/special_tokens_map.json',
 'phi-4-v1/chat_template.jinja',
 'phi-4-v1/vocab.json',
 'phi-4-v1/merges.txt',
 'phi-4-v1/added_tokens.json',
 'phi-4-v1/tokenizer.json')

In [ ]:
# @title save LoRA adapters to the HUB.
# Just LoRA adapters
if False:
    model.save_pretrained(FINETUNED_MODEL_NAME)
    tokenizer.save_pretrained(FINETUNED_MODEL_NAME)
if False:
    model.push_to_hub("hf/"+FINETUNED_MODEL_NAME)
    tokenizer.push_to_hub("hf/"+FINETUNED_MODEL_NAME)

HfHubHTTPError: (Request ID: Root=1-68e5a318-77346bfd591068256ef5fe8b;e6a21bcb-20e5-4ca3-8d93-e785039ee48a)

403 Forbidden: You don't have the rights to create a model under the namespace "hf".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

{"morfologia": "solido_polipoide", "spessore_parietale": null, "estensione_cranio_caudale": 50.0, "distanza_oai": 60.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "cavallo", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "sedi_locoregionali": {"mesorettali": true, "rettali_superiori": false, "mesenterici_inferiori": false, "iliaci_interni": false, "otturatori": false, "sacrali": false, "inguinali_sotto_dentata": false}, "sedi_non_locoregionali": {"inguinali": false, "iliaci_esterni": false, "iliaci_comuni": false, "paraortici": false, "altri": false}, "depositi_tumorali": "si", "numero_depositi": 0.0, "emvi_esteso": "si"}<|im_end|>


Per scaricare una cartella da Colab, puoi usare il comando `zip` per comprimerla e poi scaricare il file zip.

Sostituisci `nome_cartella_da_scaricare` con il nome della cartella che vuoi scaricare.

In [ ]:
!zip -r /content/phi-4-v1.zip /content/phi-4-v1

  adding: content/phi-4-v1/ (stored 0%)
  adding: content/phi-4-v1/vocab.json (deflated 56%)
  adding: content/phi-4-v1/merges.txt (deflated 50%)
  adding: content/phi-4-v1/README.md (deflated 65%)
  adding: content/phi-4-v1/special_tokens_map.json (deflated 73%)
  adding: content/phi-4-v1/tokenizer.json (deflated 80%)
  adding: content/phi-4-v1/adapter_model.safetensors (deflated 8%)
  adding: content/phi-4-v1/tokenizer_config.json (deflated 95%)
  adding: content/phi-4-v1/chat_template.jinja (deflated 65%)
  adding: content/phi-4-v1/adapter_config.json (deflated 57%)


Dopo aver eseguito il comando precedente, verrà creato un file `.zip` nella directory `/content/`. Puoi quindi scaricarlo utilizzando l'interfaccia di Colab (cliccando sull'icona della cartella a sinistra, trovando il file zip e cliccando sui tre puntini per scaricarlo) oppure usando il seguente comando shell.

In [ ]:
from google.colab import files
files.download('/content/nome_cartella_compressa.zip')

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

# Alternative di salvataggio e caricamento

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
